# Assignment 2: Transformer Language Models

In this assignment you will implement a Transformer-based language model following the **OLMo 2 architecture**, train it, and compare it to a pre-trained model.

![Olmo2 overview](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/olmo2_overview.svg)

**Task markers:**
- 🎓 Suitable for oral exam discussion
- ⚙ Pure implementation task

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

---

## Part 1: Building the Transformer Components

### Task 1.1 — MLP Layer ⚙

Implement the **SwiGLU** MLP using `hidden_size` and `intermediate_size` hyperparameters.

![SwiGLU](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/swiglu.svg)

SwiGLU uses element-wise multiplication (⊗) between two linear branches:

$$\text{MLP}(x) = \bigl(W_1 x \cdot \text{SiLU}(W_2 x)\bigr) W_3$$

All `nn.Linear` layers should use `bias=False`.

**Sanity check:** Create an untrained MLP layer. Create a 3-dimensional tensor where the last dimension equals `hidden_size`. Applying the MLP to this tensor should produce output with the same shape as the input.

In [ ]:
# Task 1.1 — SwiGLU MLP

class A2MLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        # TODO: define linear layers (bias=False)

    def forward(self, x):
        # TODO: implement SwiGLU forward pass
        pass


# Sanity check
# mlp = A2MLP(hidden_size=64, intermediate_size=128)
# x = torch.randn(2, 10, 64)
# assert mlp(x).shape == x.shape


### Task 1.2 — Normalization ⚙

Implement **Root Mean Square (RMS) layer normalization**, or use PyTorch's built-in `nn.RMSNorm`.

Configuration:
- `eps` → `rms_norm_eps`
- `normalized_shape` → hidden layer size
- `elementwise_affine=True`

**Sanity check:** Apply the same testing approach as Task 1.1.

In [ ]:
# Task 1.2 — RMS Normalization

class A2RMSNorm(nn.Module):
    def __init__(self, hidden_size, rms_norm_eps=1e-5):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        pass


# Sanity check
# norm = A2RMSNorm(hidden_size=64)
# x = torch.randn(2, 10, 64)
# assert norm(x).shape == x.shape


### Task 1.3 — Multi-Head Attention 🎓

Implement standard **multi-head attention** with **RoPE** (Rotary Position Embedding).

![MHA](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/mha.svg)

Key hyperparameters: `hidden_size`, `num_attention_heads`  
Derived: `d_h = hidden_size // num_attention_heads`

**Steps:**
1. Compute query, key, value projections
2. Reshape: `q = q.view(b, m, n_h, d_h).transpose(1, 2)`
3. Apply RoPE via the provided `apply_rotary_pos_emb` utility
4. Compute attention using `F.scaled_dot_product_attention` with `is_causal=True`
5. Project output

$$\alpha(q, k) = \frac{q \cdot k^T}{\sqrt{d_h}}$$
$$A(q, k) = \text{softmax}(\alpha(q, k) + \text{mask})$$
$$\text{Attention}(q, k, v) = A(q, k) \cdot v$$

All projection layers: `bias=False`.

**Sanity check:** Verify output shape and no crashes at multiple intermediate stages.

In [ ]:
# Provided utility — RoPE helper (do not modify)
def apply_rotary_pos_emb(q, k, cos, sin):
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [ ]:
# Task 1.3 — Multi-Head Attention with RoPE

class A2RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000):
        super().__init__()
        # TODO: precompute inverse frequencies and register as buffer

    def forward(self, x, seq_len=None):
        # TODO: return cos, sin
        pass


class A2MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_attention_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_attention_heads
        self.head_dim = hidden_size // num_attention_heads
        # TODO: define q, k, v, and output projections (bias=False)

    def forward(self, x, rotary_emb):
        # b, m, _ = x.shape
        # q = self.q_proj(x).view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
        # k = ...
        # v = ...
        # cos, sin = rotary_emb(x, seq_len=m)
        # q, k = apply_rotary_pos_emb(q, k, cos, sin)
        # attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        # attn_output = attn_output.transpose(1, 2).reshape(b, m, self.hidden_size)
        # return self.o_proj(attn_output)
        pass  # TODO


# Sanity check
# attn = A2MultiHeadAttention(hidden_size=64, num_attention_heads=4)
# rope = A2RotaryEmbedding(dim=16)
# x = torch.randn(2, 10, 64)
# assert attn(x, rope).shape == x.shape


### Task 1.4 — Full Transformer Decoder Layer 🎓

Assemble a single Transformer decoder block. In `__init__`, create:
- Multi-head attention layer
- MLP layer
- Two RMSNorm normalizers

![fullblock](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/fullblock.svg)

In `forward`, connect them with **residual connections** at the right places:

```
h_new = do_something(h_old)
out   = h_new + h_old
```

**Sanity check:** Verify correct output shapes and no crashes.

In [ ]:
# Task 1.4 — Transformer Decoder Layer

class A2TransformerLayer(nn.Module):
    def __init__(self, hidden_size, num_attention_heads, intermediate_size, rms_norm_eps=1e-5):
        super().__init__()
        # TODO: attention, mlp, norms

    def forward(self, x, rotary_emb):
        # TODO: apply norm → attention → residual, then norm → mlp → residual
        pass


# Sanity check
# layer = A2TransformerLayer(hidden_size=64, num_attention_heads=4, intermediate_size=128)
# rope = A2RotaryEmbedding(dim=16)
# x = torch.randn(2, 10, 64)
# assert layer(x, rope).shape == x.shape


### Task 1.5 — Complete Transformer Stack 🎓

Assemble the full model including:
- Token embedding layer
- Stack of Transformer decoder layers (use `nn.ModuleList`, not a plain Python list)
- Final RMSNorm
- Unembedding (LM head) layer — **no bias terms**

Create `A2RotaryEmbedding` in `__init__` and pass the rotations to each layer in `forward`.

**Sanity check:** Create a 2-dimensional *integer* tensor and apply your Transformer to it. The result should be a 3-dimensional tensor where the last dimension equals the vocabulary size.

In [ ]:
# Task 1.5 — Full Transformer Stack

class A2TransformerConfig:
    def __init__(
        self,
        vocab_size=32000,
        hidden_size=256,
        num_hidden_layers=4,
        num_attention_heads=4,
        intermediate_size=512,
        rms_norm_eps=1e-5,
        max_position_embeddings=2048,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.rms_norm_eps = rms_norm_eps
        self.max_position_embeddings = max_position_embeddings


class A2TransformerModel(nn.Module):
    def __init__(self, config: A2TransformerConfig):
        super().__init__()
        # TODO: embedding, ModuleList of layers, norm, lm_head (bias=False)

    def forward(self, input_ids, labels=None):
        # TODO
        pass


# Sanity check
# config = A2TransformerConfig()
# model = A2TransformerModel(config)
# x = torch.randint(0, config.vocab_size, (2, 20))
# out = model(x)
# assert out.shape == (2, 20, config.vocab_size)


---

## Part 2: Training

### Task 2.1 — Training the Language Model 🎓

Select suitable hyperparameters (number of Transformer layers, hidden size, number of attention heads).  
For this assignment, use a **small Transformer** (e.g. a couple of layers).

Run the training function and compute the perplexity on the validation set (same method as Assignment 1).

> **Alternative:** Use the HuggingFace `Trainer`.

In [ ]:
# Task 2.1 — Train the Transformer

# config = A2TransformerConfig(
#     vocab_size=...,
#     hidden_size=...,
#     num_hidden_layers=...,
#     num_attention_heads=...,
#     intermediate_size=...,
# )
# model = A2TransformerModel(config)

# TODO: training loop (reuse or adapt A1Trainer)


---

## Part 3: Text Generation

### Task 3.1 — Predicting the Next Word ⚙

Apply the model to an encoded prompt, extract the output at the **last position**, find the highest-scoring token index with `argmax`, and decode it using the tokenizer.

In [ ]:
# Task 3.1 — Next-word prediction

def predict_next_word(model, tokenizer, prompt):
    pass  # TODO


### Task 3.2 — Generating Texts 🎓

Implement a **random sampling** algorithm with the following parameters:
- `model`
- `prompt`
- `max_length`
- `temperature`
- `topk`

Terminate when the end-of-text symbol is produced or `max_length` steps are reached.  
Use `torch.distributions.Categorical` for sampling and `torch.topk` for top-K filtering.

Experiment with different values of `temperature` and `topk` and observe the effect on output quality.

In [ ]:
# Task 3.2 — Text generation with top-K sampling

def generate_text(model, tokenizer, prompt, max_length=100, temperature=1.0, topk=50):
    # Hint: use torch.distributions.Categorical for sampling
    # dist = torch.distributions.Categorical(logits=filtered_logits)
    # next_token = dist.sample()
    pass  # TODO


# Test with sample prompts
# print(generate_text(model, tokenizer, "The patient was diagnosed with"))
# print(generate_text(model, tokenizer, "The study found that", temperature=0.7, topk=20))


### Task 3.3 — Comparing to a Pre-trained Transformer 🎓

Load the pre-trained **OLMo-2 1B** model and compare its text generation quality to your own model.

> **Note:** This model is *not* instruction-tuned — use it as a language model only.

> **Optional:** Copy weights from the pre-trained model into your implementation to verify architectural equivalence.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'allenai/OLMo-2-0425-1B'

# Download or point to a local cache directory
local_dir = model_name   # change to local path if already downloaded

# Task 3.3 — Load pre-trained model and tokenizer
# tokenizer_pretrained = AutoTokenizer.from_pretrained(local_dir)
# model_pretrained = AutoModelForCausalLM.from_pretrained(local_dir)

# Compare generation quality
# print(generate_text(model_pretrained, tokenizer_pretrained, "The patient was diagnosed with"))
